### Imports ###

In [1]:
!pip install psycopg2

In [2]:
import pandas as pd
import numpy as np
import psycopg2.extras as extras

### Load Config ###

In [3]:
import sys
sys.path.append("..")

from config import get_connection, logger

### ETL Pipeline ###

#### Extract (E) - Process ####

In [ ]:
logger.info("Starting Companies ETL - Extract")

CSV_FILE = "../raw_data/companies.csv"
df = pd.read_csv(CSV_FILE)

logger.info(f"Extracted {len(df)} company records")
display(df.head())

#### Transformation (T) - Process ####

In [10]:
logger.info("Transforming companies data")

df.columns = df.columns.str.lower().str.replace(" ", "_")

df["company_id"] = df["company_id"].astype(int)
df["company_size"] = pd.to_numeric(df["company_size"], errors = "coerce")

df = df.drop_duplicates(subset = ["company_id"])

logger.info(f"Post-transform row count: {len(df)}")

SyntaxError: invalid syntax (4044636130.py, line 5)

#### Load (L) - Process ####

In [ ]:
logger.info("Loading companies into database")

insert_query = """
INSERT INTO companies (
    company_id, company_name, company_description,
    company_size, state, country, city,
    zip_code, address, url
)
VALUES %s
ON CONFLICT (company_id)
DO UPDATE SET
company_name = EXCLUDED.company_name,
company_description = EXCLUDED.company_description,
company_size = EXCLUDED.company_size,
state = EXCLUDED.state,
country = EXCLUDED.country,
city = EXCLUDED.city,
zip_code = EXCLUDED.zip_code,
address = EXCLUDED.address,
url = EXCLUDED.url
"""

data = [tuple(row) for row in df.values]

conn = get_connection()
try:
    with conn.cursor() as cur:
        extras.execute_values(cur, insert_query, data)
        conn.commit()
        logger.info("Companies load completed")
finally:
    conn.close()